# CSV - Missing Records

Remove scattered observations, multiple groups of consecutive records, or both.

**Instructions:** Use `prepare_data_csv_update.ipynb` to create a clean `data/weather.csv` first. This notebook overwrites it; restore clean data before rerunning. `EXPECTED_FREQUENCY="h"` verifies known hourly coverage. Set it to `None` for irregular records; without an independent record inventory, repeated removals cannot be detected reliably.

In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("../../data/weather.csv")
TIME_COLUMN = "time"
KEY_COLUMNS = ["time"]
TIMEZONE = "America/Edmonton"
START_DATE = "2025-06-01"
END_DATE = "2025-09-01"  # Exclusive; local calendar dates.
SEED = 814205

# Missing Records
RECORD_METHOD = "mixed"  # "scattered", "grouped", "mixed", "random".
RECORD_RATE = (0.01, 0.03)  # Share of current eligible records removed.
RECORD_GROUPS = 3
RECORD_GROUPED_SHARE = 0.70
EXPECTED_FREQUENCY = "h"  # Known one-record-per-hour schedule; None for irregular event data.


In [ ]:
# Shared placement helper: copy once into a combined dirty_data.py.
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen

In [ ]:
def remove_records(
    df, rng, *, time_column, key_columns, timezone, start_date, end_date,
    method, record_rate, n_groups, grouped_share, expected_frequency=None,
):
    required = [time_column, *key_columns]
    if any(column not in df for column in required):
        raise ValueError("Time or key column not found.")

    clean = df.reset_index(drop=True).copy()
    if clean[key_columns].isna().any().any() or clean.duplicated(subset=key_columns).any():
        raise ValueError("Expected unique, nonmissing keys. Restore clean data first.")

    time = pd.to_datetime(clean[time_column], utc=True, errors="raise")
    if time.isna().any():
        raise ValueError("Timestamps cannot be missing.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")

    scope = time.ge(start) & time.lt(end)
    eligible = clean.loc[scope].sort_values(
        [time_column, *[c for c in key_columns if c != time_column]], kind="stable"
    )
    if eligible.empty:
        raise ValueError("No eligible records.")

    # An expected schedule makes existing gaps (including a second run) detectable.
    if expected_frequency is not None:
        if key_columns != [time_column]:
            raise ValueError("Schedule validation requires one record per timestamp.")
        expected = pd.date_range(start, end, freq=expected_frequency, inclusive="left")
        observed = pd.DatetimeIndex(time.loc[scope])
        if not expected.equals(observed.sort_values()):
            raise ValueError("Expected timestamp coverage is already incomplete or irregular. Restore clean data first.")

    selected, _ = select_positions(
        len(eligible), rng, method=method, rate=record_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )

    # Preserve every surviving original record, without manufacturing nulls or copies.
    removed_index = eligible.index[selected]
    return clean.drop(index=removed_index).reset_index(drop=True)


In [ ]:
# Small business-like trial: irregular order timestamps, including shared timestamps.
orders = pd.DataFrame({
    "time": [pd.Timestamp("2025-07-01", tz=TIMEZONE).tz_convert("UTC") + pd.Timedelta(hours=(i // 2) * 5) for i in range(60)],
    "order_id": range(1000, 1060),
    "amount": [float(i * 3 + 10) for i in range(60)],
})
trial = remove_records(
    orders, Random(SEED), time_column="time", key_columns=["time", "order_id"],
    timezone=TIMEZONE, start_date="2025-07-01", end_date="2025-08-01",
    method="mixed", record_rate=(0.20, 0.20), n_groups=3,
    grouped_share=0.70, expected_frequency=None,
)
assert len(orders) - len(trial) == 12
assert trial["order_id"].isin(orders["order_id"]).all()
assert trial["order_id"].is_unique
assert len(remove_records(
    orders, Random(SEED), time_column="time", key_columns=["time", "order_id"],
    timezone=TIMEZONE, start_date="2025-07-01", end_date="2025-08-01",
    method="mixed", record_rate=(0.20, 0.20), n_groups=3,
    grouped_share=0.70, expected_frequency=None,
)) == len(trial)
trial.head()


In [ ]:
# Apply to a freshly prepared CSV; overwrite weather.csv.
clean = pd.read_csv(DATA_PATH)
dirty = remove_records(
    clean, Random(SEED), time_column=TIME_COLUMN, key_columns=KEY_COLUMNS,
    timezone=TIMEZONE, start_date=START_DATE, end_date=END_DATE,
    method=RECORD_METHOD, record_rate=RECORD_RATE, n_groups=RECORD_GROUPS,
    grouped_share=RECORD_GROUPED_SHARE, expected_frequency=EXPECTED_FREQUENCY,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)


In [ ]:
# Inspect: rows can be missing without any null values.
clean.shape, saved.shape, saved.isna().any().any()


In [ ]:
# Inspect coverage when a known regular schedule exists.
if EXPECTED_FREQUENCY is not None:
    start = pd.Timestamp(START_DATE, tz=TIMEZONE).tz_convert("UTC")
    end = pd.Timestamp(END_DATE, tz=TIMEZONE).tz_convert("UTC")
    expected = pd.date_range(start, end, freq=EXPECTED_FREQUENCY, inclusive="left")
    observed = pd.DatetimeIndex(pd.to_datetime(saved[TIME_COLUMN], utc=True))
    missing_times = expected.difference(observed)
    print("Missing expected timestamps:", len(missing_times))
    display(missing_times[:20])


In [ ]:
# Verify exactly which original records disappeared after CSV round-trip.
comparison = clean[KEY_COLUMNS].merge(saved[KEY_COLUMNS], on=KEY_COLUMNS, how="left", indicator=True)
removed = clean.loc[comparison["_merge"].eq("left_only")].copy()
assert len(removed) == len(clean) - len(saved)
assert len(removed) > 0
pd.testing.assert_frame_equal(
    saved, clean.drop(index=removed.index).reset_index(drop=True),
    check_exact=False, rtol=1e-12,
)
print("Removed records:", len(removed))
removed.head(20)


In [ ]:
import matplotlib.pyplot as plt

# Plot actual removed records per local date (developer-side truth).
removed_dates = pd.to_datetime(removed[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
all_dates = pd.date_range(START_DATE, END_DATE, freq="D", inclusive="left").date
removed_daily = removed_dates.value_counts().reindex(all_dates, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(pd.to_datetime(removed_daily.index), removed_daily.values, width=0.9)
ax.set_title("Removed records by local date")
ax.set_xlabel("Local date")
ax.set_ylabel("Records removed")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
